# Análise dos dados da CPA/UFT — ciclo 2025

Pipeline completo de análise exploratória dos dados da Comissão Própria de Avaliação da UFT — ciclo 2025, com 2024 como termo histórico de comparação. As funções analíticas vivem em [`analise_dados_2025.py`](./analise_dados_2025.py); este notebook orquestra os seis estágios do pipeline, expõe os artefatos intermediários para inspeção e gera as figuras usadas no TCC, nos slides de defesa e no kit relatório CPA.

**Estágios:**

1. Carregamento e consolidação — base bruta → 649 respondentes únicos
2. Estatística descritiva — por questão, eixo, NSO, campus, curso
3. Correlações e consistência interna — Spearman + Cronbach
4. Análise textual da q70 — frequência, bigramas, categorização
5. Comparação temporal 2024 → 2025 — variações por questão
6. Geração de figuras — TCC (FT e Resultados), slides e kit relatório CPA

## Setup

In [ ]:
%matplotlib inline

from pathlib import Path

import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

import analise_dados_2025 as cpa

FIG_DIR = cpa.FIG_DIR
KIT_DIR = FIG_DIR / "kit_cpa"
SLIDES_IMG = Path(r"c:/Users/emanu/Downloads/slides_tcc/img")

## Estágio 1 — Carregamento e consolidação

Lê a base bruta de 2025, aplica a regra de prioridade `Doc > Téc > Disc > Egr` (Hurlbert, 1984), determina o campus a partir da coluna do segmento consolidado, normaliza colunas Likert/binárias/condicionais e adiciona os scores compostos por eixo. A base sai de 657 → 649 respondentes únicos.

In [ ]:
df = cpa.carregar_e_consolidar()

In [ ]:
print(f"Forma: {df.shape}")
print("\nDistribuição por segmento:")
print(df["segmento"].value_counts())
print("\nDistribuição por campus:")
print(df["campus"].value_counts())

## Estágio 2 — Estatística descritiva

Sete saídas: por questão, por eixo, taxa de NSO (total e por segmento), por campus, campus × segmento, e por curso (discentes com N ≥ 10). Todos os DataFrames são salvos em `outputs/2025/`.

In [ ]:
desc = cpa.descritiva(df)

In [ ]:
# Top 15 questões por média
desc["por_questao"].head(15)

In [ ]:
# Médias por eixo SINAES
desc["por_eixo"]

In [ ]:
# Top 12 questões com maior taxa de NSO ("não sei opinar")
desc["nso"].head(12)

In [ ]:
# Médias por eixo × campus (N ≥ 30)
desc["por_campus"]

In [ ]:
# Médias por eixo × curso (discentes, N ≥ 10)
desc["por_curso"]

## Estágio 3 — Correlações e consistência interna

Spearman inter-eixos, Spearman questão-a-questão (33 questões universais), análise estratificada por segmento (Discente/Docente/Técnico), top pares por |ρ|, e alfa de Cronbach por eixo.

In [ ]:
inf = cpa.inferencial(df)

In [ ]:
# Correlações Spearman entre os 4 eixos Likert
inf["spearman_eixos"]

In [ ]:
# Top 15 pares de questões por |ρ|
inf["top_pares"].head(15)

In [ ]:
# Estratificação por segmento — top 10 pares com rho por segmento
inf["estratificacao_por_segmento"].head(10)

In [ ]:
# Cronbach alpha por eixo (consistência interna)
inf["cronbach"]

## Estágio 4 — Análise textual da q70

A questão q70 — "a voz da comunidade" — é o único campo aberto. Aqui: estatísticas básicas do corpus, frequência de termos, bigramas e categorização por eixo SINAES via dicionário temático (Bardin, 1977).

In [ ]:
text = cpa.textual(df)

In [ ]:
# Estatísticas do corpus
text["estat"]

In [ ]:
# Top 20 termos mais frequentes
text["top_termos"].head(20)

In [ ]:
# Top 15 bigramas
text["top_bigramas"].head(15)

In [ ]:
# Categorização dos comentários por eixo SINAES
text["categorizacao"]

## Estágio 5 — Comparação temporal 2024 → 2025

Carrega a base de 2024, aplica o mesmo pipeline de consolidação (com o catálogo de eixos reconstruído a partir do mapeamento manual de 51 questões comparáveis) e calcula as variações por questão e as médias por eixo de cada ano.

In [ ]:
tmp = cpa.temporal(df, mapa=inf["mapeamento"])

In [ ]:
# Top 10 maiores variações entre 2024 e 2025
tmp["comparacao"].head(10)

In [ ]:
# Médias por eixo em cada ano
import pandas as pd
pd.DataFrame({"2024": tmp["medias_2024"], "2025": tmp["medias_2025"]})

## Estágio 6 — Geração de figuras

Roda todas as figuras do TCC (Fundamentação Teórica e Resultados), as figuras dos slides de defesa, e o kit relatório CPA. Os PNGs (e PDFs correspondentes) são salvos em `figuras/2025/` e `figuras/2025/kit_cpa/`.

In [ ]:
ctx = {
    "desc_e": desc["por_eixo"],
    "rho": inf["spearman_eixos"],
    "rho_q": inf["spearman_questoes"],
    "rho_q_estratificado": inf["spearman_questoes_estratificado"],
    "df_2024": tmp["df_2024"],
    "med_2024": tmp["medias_2024"],
    "med_2025": tmp["medias_2025"],
    "top_termos": text["top_termos"],
    "mapa": inf["mapeamento"],
}
figs = cpa.figuras(df, ctx)
print(f"\n{len(figs)} figuras geradas em {FIG_DIR}")

### Figuras da Fundamentação Teórica (didáticas)

Exemplos de boxplot, histograma, heatmap e nuvem de palavras gerados sobre os próprios dados da CPA 2025 — substituem ilustrações genéricas no capítulo de FT, mantendo coerência visual com os Resultados.

In [ ]:
didaticas = [
    "fig_exemplo_boxplot_didatico_2025.png",
    "fig_exemplo_histograma_2025.png",
    "fig_exemplo_heatmap_2025.png",
    "nuvem_q70_2025.png",
]
for nome in didaticas:
    p = FIG_DIR / nome
    if p.exists():
        print(nome)
        display(Image(str(p))) 

### Figuras dos Resultados do TCC

Painel analítico completo: distribuição por eixo (média e boxplot), variação por segmento (radar + heatmaps eixos 2-3 e 4-5), variação por campus (radar + heatmap por questão), cluster Spearman (matriz inter-eixos + pares estratificados estável e heterogêneo), comparação temporal 2024 → 2025 (radar e gap).

In [ ]:
resultados = [
    "media_eixos_2025.png",
    "boxplot_eixos_2025.png",
    "radar_segmentos_2025.png",
    "heatmap_segmento_eixos2_3_2025.png",
    "heatmap_segmento_eixos4_5_2025.png",
    "radar_campus_2025.png",
    "heatmap_questao_campus_2025.png",
    "heatmap_spearman_eixos_2025.png",
    "fig_par_estavel_2025.png",
    "fig_par_heterogeneo_2025.png",
    "radar_2024_vs_2025.png",
    "gap_eixos_2024_2025.png",
]
for nome in resultados:
    p = FIG_DIR / nome
    if p.exists():
        print(nome)
        display(Image(str(p))) 

### Figuras dos slides de defesa

Subconjunto reaproveitado dos Resultados, mais três figuras específicas dos slides geradas por scripts *standalone* em [`slides_tcc/`](file:///c:/Users/emanu/Downloads/slides_tcc/): boxplot estratificado por campus, distribuição segmento × campus e top-12 gaps Disc-Doc. Essas três ficaram fora do pipeline principal nesta versão; ficam registradas como dívida técnica a portar para `analise_dados_2025.py` num próximo ciclo.

In [ ]:
slides = [
    "radar_campus_2025.png",
    "heatmap_segmento_eixos4_5_2025.png",
    "gap_disc_doc_top12_2025.png",          # standalone: gerar_gap_disc_doc_top.py
    "boxplot_eixos_campus_2025.png",        # standalone: gerar_boxplot_campus.py
    "distribuicao_segmento_campus_2025.png",# standalone: gerar_distribuicao_segmento_campus.py
    "fig_par_estavel_2025.png",
    "fig_par_heterogeneo_2025.png",
    "radar_2024_vs_2025.png",
    "nuvem_q70_2025.png",
]
for nome in slides:
    p = SLIDES_IMG / nome
    if p.exists():
        print(nome)
        display(Image(str(p))) 
    else:
        print(f"(ausente em slides_tcc/img/) {nome}")

### Kit relatório CPA

Conjunto de figuras adicionais entregue à CPA para uso direto no relatório anual: tabela campus × segmento, participação por segmento 2024 vs 2025, distribuição Likert percentual por eixo (com painéis até 4 questões) e por campus, setores para questões binárias (Sim/Não/NSO), médias por eixo × segmento e × campus.

In [ ]:
ancoras_kit = [
    "participacao_segmento_2024_2025.png",
    "media_eixos_por_segmento_2025.png",
    "media_eixos_por_campus_2025.png",
    "likert_segmento_eixo1_2025.png",
    "likert_campus_eixo1_2025.png",
    "setores_q1_2025.png",
    "setores_q28_2025.png",
]
for nome in ancoras_kit:
    p = KIT_DIR / nome
    if p.exists():
        print(nome)
        display(Image(str(p))) 

total_kit = len(list(KIT_DIR.glob("*.png")))
print(f"\nKit CPA — {total_kit} figuras no total em {KIT_DIR}")

## Síntese e reprodutibilidade

O pipeline em estágios deixa o notebook modular: cada célula corresponde a um movimento do método; os artefatos (DataFrames descritivos, matrizes de correlação, médias temporais, contagens textuais) ficam disponíveis em variáveis nomeadas para extração no LaTeX; e as figuras são salvas em `figuras/2025/` (TCC, slides) e `figuras/2025/kit_cpa/` (kit).

**Para rodar tudo de uma vez** (sem inspeção intermediária):

```python
artefatos = cpa.pipeline_2025()
```

**Para rodar via linha de comando** (execução headless, backend Agg):

```bash
python analise_dados_2025.py
```